# ============================================================
# Stage-aware Evaluation Script
# Part 1: Reproducibility variance (stage == "reproducibility")
# Part 2: Ablation accuracy comparison vs Full model
#
# Works for BOTH:
#   - Point forecasts: *_single columns
#   - Multi-step forecasts: *_series columns (list-like strings)
#
# Outputs:
#   - reproducibility_point_query_level.csv
#   - reproducibility_multi_query_level.csv
#   - ablation_point_stage_metrics.csv
#   - ablation_multi_stage_metrics.csv
#   - ablation_point_stage_deltas.csv
#   - ablation_multi_stage_deltas.csv
# ============================================================

In [ ]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path

EPS = 1e-9

# ----------------------------
# CONFIG: set your paths here
# ----------------------------
MODEL_NAME = "OpenAI"  # for labeling outputs (e.g., Claude/Gemini/OpenAI/DeepSeek)

POINT_PATH = "point/extracted/OpenAI_point_forecast_extracted.csv"
MULTI_PATH = "multi/extracted/OpenAI_multi_step_extracted.csv"

OUT_DIR = "Metrics/"+MODEL_NAME+"/"   # folder where outputs are saved
SEED_FOR_ABLATION = 7   # ablation comparison uses a single fixed seed

# Column names (adjust if yours differ)
COL_STAGE = "stage"
COL_QUERY = "query_id"
COL_SEED  = "seed"      # if your file uses another name, change it here

# Point columns
GT_TD_SINGLE = "gt_TOTALDEMAND_single"
PR_TD_SINGLE = "pred_TOTALDEMAND_single"
GT_RRP_SINGLE = "gt_RRP_single"
PR_RRP_SINGLE = "pred_RRP_single"

# Multi columns (strings that look like Python lists)
GT_TD_SER = "gt_TOTALDEMAND_series"
PR_TD_SER = "pred_TOTALDEMAND_series"
GT_RRP_SER = "gt_RRP_series"
PR_RRP_SER = "pred_RRP_series"


# ============================
# Helpers
# ============================
def safe_float_series(s: pd.Series) -> pd.Series:
    """Convert to numeric safely; coercing invalid to NaN."""
    return pd.to_numeric(s, errors="coerce")


def parse_list_cell(x):
    """
    Parse a cell that contains a Python-list-like string.
    Returns a list of floats, or [] if parsing fails.
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [float(v) for v in x]
    if isinstance(x, (np.ndarray, tuple)):
        return [float(v) for v in list(x)]
    try:
        out = ast.literal_eval(x)
        if isinstance(out, list):
            return [float(v) for v in out]
        return []
    except Exception:
        return []


def compute_accuracy_metrics(y_true, yhat):
    """
    Returns MAE, RMSE, Mean_y, NMAE, NRMSE
    (Mean-normalized errors; robust and scale-aware.)
    """
    y_true = np.asarray(y_true, dtype=float)
    yhat   = np.asarray(yhat, dtype=float)

    mae  = float(np.mean(np.abs(y_true - yhat)))
    rmse = float(np.sqrt(np.mean((y_true - yhat) ** 2)))
    mean_y = float(np.mean(y_true))

    nmae  = float(mae  / (mean_y + EPS))
    nrmse = float(rmse / (mean_y + EPS))

    return {"MAE": mae, "RMSE": rmse, "Mean_y": mean_y, "NMAE": nmae, "NRMSE": nrmse}


def compute_repro_stats(preds):
    """
    Reproducibility stats over repeated runs of SAME query:
      mu, sigma (ddof=1), CV = sigma/|mu|
    """
    preds = np.asarray(preds, dtype=float)
    if len(preds) < 2:
        return None
    mu = float(np.mean(preds))
    sigma = float(np.std(preds, ddof=1))
    cv = float(sigma / (abs(mu) + EPS))
    return {"mean_pred": mu, "std_pred": sigma, "CV": cv, "num_runs": int(len(preds))}


# ============================
# Part 1: Reproducibility variance
# ============================
def repro_variance_point(df_point: pd.DataFrame) -> pd.DataFrame:
    """
    For stage == reproducibility:
      - compute variance of predictions across seeds per query
      - for TOTALDEMAND and RRP separately
    """
    df = df_point[df_point[COL_STAGE] == "reproducibility"].copy()

    # Ensure numeric
    for c in [PR_TD_SINGLE, PR_RRP_SINGLE]:
        if c in df.columns:
            df[c] = safe_float_series(df[c])

    rows = []
    for query_id, g in df.groupby(COL_QUERY):
        # TOTALDEMAND
        if PR_TD_SINGLE in g.columns:
            preds = g[PR_TD_SINGLE].dropna().values
            st = compute_repro_stats(preds)
            if st:
                rows.append({
                    "Model": MODEL_NAME, "Type": "POINT", "Target": "TOTALDEMAND",
                    "query_id": query_id, **st
                })

        # RRP
        if PR_RRP_SINGLE in g.columns:
            preds = g[PR_RRP_SINGLE].dropna().values
            st = compute_repro_stats(preds)
            if st:
                rows.append({
                    "Model": MODEL_NAME, "Type": "POINT", "Target": "RRP",
                    "query_id": query_id, **st
                })

    return pd.DataFrame(rows)


def repro_variance_multi(df_multi: pd.DataFrame) -> pd.DataFrame:
    """
    For stage == reproducibility:
      - per query_id, we have multiple series (different seeds)
      - compute per-timestep sigma, CV then summarize:
          mean_std, mean_cv, max_std, max_cv, num_runs, horizon_len
      - for TOTALDEMAND and RRP separately
    """
    df = df_multi[df_multi[COL_STAGE] == "reproducibility"].copy()

    rows = []

    for query_id, g in df.groupby(COL_QUERY):
        # TOTALDEMAND series
        if PR_TD_SER in g.columns:
            series_list = [np.array(parse_list_cell(x), dtype=float) for x in g[PR_TD_SER].tolist()]
            series_list = [s for s in series_list if len(s) > 0]
            if len(series_list) >= 2:
                L = min(len(s) for s in series_list)
                mat = np.vstack([s[:L] for s in series_list])  # (runs, horizon)
                mu = np.mean(mat, axis=0)
                sigma = np.std(mat, axis=0, ddof=1)
                cv = sigma / (np.abs(mu) + EPS)
                rows.append({
                    "Model": MODEL_NAME, "Type": "MULTI", "Target": "TOTALDEMAND",
                    "query_id": query_id,
                    "num_runs": int(mat.shape[0]),
                    "horizon_len": int(L),
                    "mean_std": float(np.mean(sigma)),
                    "mean_CV": float(np.mean(cv)),
                    "max_std": float(np.max(sigma)),
                    "max_CV": float(np.max(cv)),
                })

        # RRP series
        if PR_RRP_SER in g.columns:
            series_list = [np.array(parse_list_cell(x), dtype=float) for x in g[PR_RRP_SER].tolist()]
            series_list = [s for s in series_list if len(s) > 0]
            if len(series_list) >= 2:
                L = min(len(s) for s in series_list)
                mat = np.vstack([s[:L] for s in series_list])
                mu = np.mean(mat, axis=0)
                sigma = np.std(mat, axis=0, ddof=1)
                cv = sigma / (np.abs(mu) + EPS)
                rows.append({
                    "Model": MODEL_NAME, "Type": "MULTI", "Target": "RRP",
                    "query_id": query_id,
                    "num_runs": int(mat.shape[0]),
                    "horizon_len": int(L),
                    "mean_std": float(np.mean(sigma)),
                    "mean_CV": float(np.mean(cv)),
                    "max_std": float(np.max(sigma)),
                    "max_CV": float(np.max(cv)),
                })

    return pd.DataFrame(rows)


# ============================
# Part 2: Ablation accuracy (stage != reproducibility) vs Full
# ============================
def ablation_point_stage_metrics(df_point: pd.DataFrame) -> pd.DataFrame:
    """
    Compare accuracy per stage using a single fixed seed (SEED_FOR_ABLATION).
    Stage == reproducibility => Full model
    Stage != reproducibility => Ablated component
    """
    df = df_point.copy()

    # Filter by seed if available
    if COL_SEED in df.columns:
        df = df[df[COL_SEED] == SEED_FOR_ABLATION].copy()

    # Ensure numeric
    for c in [GT_TD_SINGLE, PR_TD_SINGLE, GT_RRP_SINGLE, PR_RRP_SINGLE]:
        if c in df.columns:
            df[c] = safe_float_series(df[c])

    out_rows = []
    for stage, g in df.groupby(COL_STAGE):
        # TOTALDEMAND
        m = g[GT_TD_SINGLE].notna() & g[PR_TD_SINGLE].notna()
        if m.any():
            met = compute_accuracy_metrics(g.loc[m, GT_TD_SINGLE], g.loc[m, PR_TD_SINGLE])
            out_rows.append({"Model": MODEL_NAME, "Type": "POINT", "Stage": stage, "Target": "TOTALDEMAND", **met})

        # RRP
        m = g[GT_RRP_SINGLE].notna() & g[PR_RRP_SINGLE].notna()
        if m.any():
            met = compute_accuracy_metrics(g.loc[m, GT_RRP_SINGLE], g.loc[m, PR_RRP_SINGLE])
            out_rows.append({"Model": MODEL_NAME, "Type": "POINT", "Stage": stage, "Target": "RRP", **met})

    return pd.DataFrame(out_rows)


def ablation_multi_stage_metrics(df_multi: pd.DataFrame) -> pd.DataFrame:
    """
    Compute multi-step accuracy per stage using a single fixed seed (SEED_FOR_ABLATION).
    We flatten all series values for that stage (after aligning lengths per-row).
    """
    df = df_multi.copy()

    if COL_SEED in df.columns:
        df = df[df[COL_SEED] == SEED_FOR_ABLATION].copy()

    out_rows = []
    for stage, g in df.groupby(COL_STAGE):
        # Flatten within stage, align per row by min length
        td_true_all, td_pred_all = [], []
        rrp_true_all, rrp_pred_all = [], []

        for _, row in g.iterrows():
            td_true = parse_list_cell(row.get(GT_TD_SER, np.nan))
            td_pred = parse_list_cell(row.get(PR_TD_SER, np.nan))
            rrp_true = parse_list_cell(row.get(GT_RRP_SER, np.nan))
            rrp_pred = parse_list_cell(row.get(PR_RRP_SER, np.nan))

            if len(td_true) and len(td_pred):
                L = min(len(td_true), len(td_pred))
                td_true_all.extend(td_true[:L])
                td_pred_all.extend(td_pred[:L])

            if len(rrp_true) and len(rrp_pred):
                L = min(len(rrp_true), len(rrp_pred))
                rrp_true_all.extend(rrp_true[:L])
                rrp_pred_all.extend(rrp_pred[:L])

        if len(td_true_all) > 0:
            met = compute_accuracy_metrics(td_true_all, td_pred_all)
            out_rows.append({"Model": MODEL_NAME, "Type": "MULTI", "Stage": stage, "Target": "TOTALDEMAND", **met})

        if len(rrp_true_all) > 0:
            met = compute_accuracy_metrics(rrp_true_all, rrp_pred_all)
            out_rows.append({"Model": MODEL_NAME, "Type": "MULTI", "Stage": stage, "Target": "RRP", **met})

    return pd.DataFrame(out_rows)


def add_deltas_vs_full(stage_metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds deltas vs Full model, where Full = stage == reproducibility.
    Delta is computed for each (Type, Target) separately.
    """
    df = stage_metrics_df.copy()

    full = df[df["Stage"] == "reproducibility"][["Type", "Target", "NMAE", "NRMSE"]].rename(
        columns={"NMAE": "Full_NMAE", "NRMSE": "Full_NRMSE"}
    )

    merged = df.merge(full, on=["Type", "Target"], how="left")

    merged["Delta_NMAE"]  = merged["NMAE"]  - merged["Full_NMAE"]
    merged["Delta_NRMSE"] = merged["NRMSE"] - merged["Full_NRMSE"]

    return merged


# ============================
# Main
# ============================
def main():
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    df_point = pd.read_csv(POINT_PATH)
    df_multi = pd.read_csv(MULTI_PATH)

    # -------- Part 1: reproducibility variance --------
    repro_point = repro_variance_point(df_point)
    repro_multi = repro_variance_multi(df_multi)

    repro_point_path = out_dir / f"{MODEL_NAME}_reproducibility_point_query_level.csv"
    repro_multi_path = out_dir / f"{MODEL_NAME}_reproducibility_multi_query_level.csv"

    repro_point.to_csv(repro_point_path, index=False)
    repro_multi.to_csv(repro_multi_path, index=False)

    # -------- Part 2: ablation stage metrics (seed=7) --------
    abl_point = ablation_point_stage_metrics(df_point)
    abl_multi = ablation_multi_stage_metrics(df_multi)

    abl_point_path = out_dir / f"{MODEL_NAME}_ablation_point_stage_metrics.csv"
    abl_multi_path = out_dir / f"{MODEL_NAME}_ablation_multi_stage_metrics.csv"

    abl_point.to_csv(abl_point_path, index=False)
    abl_multi.to_csv(abl_multi_path, index=False)

    # -------- Deltas vs Full (stage=reproducibility) --------
    deltas_point = add_deltas_vs_full(abl_point)
    deltas_multi = add_deltas_vs_full(abl_multi)

    deltas_point_path = out_dir / f"{MODEL_NAME}_ablation_point_stage_deltas.csv"
    deltas_multi_path = out_dir / f"{MODEL_NAME}_ablation_multi_stage_deltas.csv"

    deltas_point.to_csv(deltas_point_path, index=False)
    deltas_multi.to_csv(deltas_multi_path, index=False)

    # Console summary
    print("Saved:")
    print("  Part1 point repro:", repro_point_path)
    print("  Part1 multi repro:", repro_multi_path)
    print("  Part2 point stage:", abl_point_path)
    print("  Part2 multi stage:", abl_multi_path)
    print("  Part2 point deltas:", deltas_point_path)
    print("  Part2 multi deltas:", deltas_multi_path)

    # Quick peek
    print("\n--- Repro Point (head) ---")
    print(repro_point.head(8))
    print("\n--- Repro Multi (head) ---")
    print(repro_multi.head(8))
    print("\n--- Ablation Point (sorted by Target, Stage) ---")
    print(abl_point.sort_values(["Target","Stage"]).head(20))
    print("\n--- Ablation Multi (sorted by Target, Stage) ---")
    print(abl_multi.sort_values(["Target","Stage"]).head(20))


if __name__ == "__main__":
    main()


Saved:
  Part1 point repro: Metrics/OpenAI/OpenAI_reproducibility_point_query_level.csv
  Part1 multi repro: Metrics/OpenAI/OpenAI_reproducibility_multi_query_level.csv
  Part2 point stage: Metrics/OpenAI/OpenAI_ablation_point_stage_metrics.csv
  Part2 multi stage: Metrics/OpenAI/OpenAI_ablation_multi_stage_metrics.csv
  Part2 point deltas: Metrics/OpenAI/OpenAI_ablation_point_stage_deltas.csv
  Part2 multi deltas: Metrics/OpenAI/OpenAI_ablation_multi_stage_deltas.csv

--- Repro Point (head) ---
    Model   Type       Target            query_id  mean_pred     std_pred  \
0  OpenAI  POINT  TOTALDEMAND  long_term_QLD1_007   5955.530    53.171421   
1  OpenAI  POINT          RRP  long_term_QLD1_007     54.600     0.313050   
2  OpenAI  POINT  TOTALDEMAND  long_term_QLD1_009   6328.714   136.472484   
3  OpenAI  POINT          RRP  long_term_QLD1_009     98.918     6.633440   
4  OpenAI  POINT  TOTALDEMAND  long_term_TAS1_010   4943.002  2058.410726   
5  OpenAI  POINT          RRP  long_t